## Environment Setup

In [1]:
!pip install open3d

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
DATASET_PATH = "/content/drive/MyDrive/THESIS_dataset/mmw/rsu_1"
MYDATASET_LIDAR_PATH = "/content/drive/MyDrive/THESIS_dataset/mmw/MyDataset_rsu1/lidar"
MYDATASET_LIDAR_LABEL_PATH = "/content/drive/MyDrive/THESIS_dataset/mmw/MyDataset_rsu1/labels_lidar"
MYDATASET_RADAR_PATH = "/content/drive/MyDrive/THESIS_dataset/mmw/MyDataset_rsu1/radar"
MYDATASET_CAR_CORNDERS_PATH = "/content/drive/MyDrive/THESIS_dataset/mmw/MyDataset_rsu1/car_box"

In [4]:
import sys
sys.path.append('/content/drive/MyDrive/THESIS/code')

import numpy as np
import open3d as o3d
import json
import os
from sklearn.neighbors import KDTree
from local_visualize.utils.get_rotation_matrix import get_rotation_matrix
from local_visualize.utils.create_bbox_lineset import create_bbox_lineset
from local_visualize.utils.car_box2label import is_car
from local_visualize.utils.pole_box2label import is_pole

## generate radar+label dataset
(x, y, z, v, label)
txt file

In [5]:
# radar location and angle
radar_loc = [1.5, 4.670000076293945, 4.099999904632568]
radar_rot = [0.0, 0.0, -135.0]

In [12]:
def radar_label_GenSingle(index):
  radar_json_path = os.path.join(DATASET_PATH, index + ".json")
  radar_txt_path = os.path.join(MYDATASET_RADAR_PATH, index + ".txt")

  # xyzv (N, 4)
  with open(radar_json_path, "r") as f:
      points = json.load(f)

  xyz_ls = []
  v_ls = []
  for p in points:
      r, az, alt, v = p["depth"], p["azimuth"], p["altitude"], p["velocity"]
      x = r * np.cos(alt) * np.cos(az)
      y = r * np.cos(alt) * np.sin(az)
      z = r * np.sin(alt)
      xyz_ls.append([x, y, z])
      v_ls.append([v])

  xyz_ls = np.array(xyz_ls)
  v_ls = np.array(v_ls)

  # carlibrate coordinates
  t = np.array(radar_loc)
  R = get_rotation_matrix(radar_rot)
  xyz_ls = (R @ xyz_ls.T).T + t

  # create label vector for each point
  label_ls = -1 * np.ones(xyz_ls.shape[0], dtype=int)

  # ==================================================
  #   buildings
  # ==================================================
  # label buildings (lidar point cloud labels --> radar point cloud labels)

  # get lidar points
  lidar_pcd_path = os.path.join(MYDATASET_LIDAR_PATH, index + ".pcd")
  lidar_pcd = o3d.io.read_point_cloud(lidar_pcd_path)
  lidar_points = np.asarray(lidar_pcd.points)
  # get lidar labels
  lidar_label_path = os.path.join(MYDATASET_LIDAR_LABEL_PATH, index + ".npy")
  lidar_labels = np.load(lidar_label_path)
  lidar_labels[lidar_labels != 1] = -1

  tree = KDTree(lidar_points)
  dist, ind = tree.query(xyz_ls, k=5)  # ind.shape = (num_radar_points, 5)
  dist_threshold = 10

  for i in range(xyz_ls.shape[0]):
    close_lidar_labels = lidar_labels[ind[i]].copy()
    close_lidar_labels = close_lidar_labels[dist[i]<dist_threshold]
    if len(close_lidar_labels) == 0:
      label_ls[i] = -1
    else:
      close_lidar_labels += 1
      label_ls[i] = np.bincount(close_lidar_labels).argmax() - 1

  # ==================================================
  #   car
  # ==================================================
  # import car box corners and create line set
  car_corners_list_path = os.path.join(MYDATASET_CAR_CORNDERS_PATH, index + ".npy")
  car_corners_list = np.load(car_corners_list_path)

  # delete boxes that are out of sight
  car_box_centers_ls = car_corners_list.mean(axis=1)
  mask = (car_box_centers_ls[:,0] < 0) & (car_box_centers_ls[:,1] < 0)
  car_corners_list = car_corners_list[mask]
  # generate box line set
  bbox_list = create_bbox_lineset(car_corners_list)

  # ==================================================
  #   pole
  # ==================================================
  # pole space range
  pole_range_box = np.array([[-12, -10, -11, -9, 0, 7],
                            [1, 3, -14, -12, 0, 7],
                            [-6, 3, -16, -12, 5, 9],
                            [-14, -10, -11, 3, 5, 9]])

  # label points as car within the car boxes
  for i, point in enumerate(xyz_ls):
      if is_car(point, car_corners_list):
          label_ls[i] = 0
      elif is_pole(point, pole_range_box):
          label_ls[i] = 2

  #%%
  # save it to txt file
  # (x, y, z, v, label)   xyz after transform
  label_ls = label_ls.reshape(-1,1)
  radar_points_labels = np.hstack((xyz_ls, v_ls, label_ls))

  radar_txt_path = os.path.join(MYDATASET_RADAR_PATH, index + ".txt")
  np.savetxt(radar_txt_path, radar_points_labels, fmt='%.6f', delimiter=',')

In [13]:
index_range = ["016653", "017853"]
index = index_range[0]

counter = 0
while index != index_range[1]:
    radar_label_GenSingle(index)
    index = f"{int(index) + 1:06d}"

    if counter % 10 == 0:
        print(index)
    counter += 1

016654
016664
016674
016684
016694
016704
016714
016724
016734
016744
016754
016764
016774
016784
016794
016804
016814
016824
016834
016844
016854
016864
016874
016884
016894
016904
016914
016924
016934
016944
016954
016964
016974
016984
016994
017004
017014
017024
017034
017044
017054
017064
017074
017084
017094
017104
017114
017124
017134
017144
017154
017164
017174
017184
017194
017204
017214
017224
017234
017244
017254
017264
017274
017284
017294
017304
017314
017324
017334
017344
017354
017364
017374
017384
017394
017404
017414
017424
017434
017444
017454
017464
017474
017484
017494
017504
017514
017524
017534
017544
017554
017564
017574
017584
017594
017604
017614
017624
017634
017644
017654
017664
017674
017684
017694
017704
017714
017724
017734
017744
017754
017764
017774
017784
017794
017804
017814
017824
017834
017844
